# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# RGB CNN benchmarking on Kather-2016

This notebook implements the whole-image benchmarks described in `Report-199_Spring.pdf`, independently of MaskCut:

1. Train a reproducible CNN from scratch on RGB histology tiles.
2. Measure performance with 500, 1,000, 1,500, and all grouped-training images.
3. Compare the scratch CNN with ImageNet-pretrained ResNet-18.

All configurations use the same source-case-grouped train, validation, and test split.

In [ ]:
from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

sns.set_theme(style="whitegrid")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
DATASET_DIR = (
    PROJECT_ROOT
    / "Colorectal Histology MNIST"
    / "Kather_texture_2016_image_tiles_5000"
    / "Kather_texture_2016_image_tiles_5000"
)
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "cnn_benchmark"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
PLOT_DIR = ARTIFACT_DIR / "plots"
for directory in (ARTIFACT_DIR, CHECKPOINT_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_DIR)
print("Artifacts:", ARTIFACT_DIR)

In [ ]:
from Methods.BaselineCNN import discover_images, make_grouped_split, set_seed
from Methods.CNNBenchmark import (
    balanced_training_subset,
    build_report_cnn,
    build_resnet18,
    count_parameters,
    run_benchmark,
)

SEED = 41
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SCRATCH_EPOCHS = 40
SCRATCH_BATCH_SIZE = 64
SCRATCH_LEARNING_RATE = 1e-3
RESNET_EPOCHS = 20
RESNET_BATCH_SIZE = 32
RESNET_LEARNING_RATE = 1e-4
set_seed(SEED)
DEVICE

## 1. Dataset manifest and grouped split

The report did not specify source-case separation. This implementation uses the case IDs embedded in filenames, preventing tiles from the same colorectal sample from appearing in multiple splits.

In [ ]:
manifest = make_grouped_split(discover_images(DATASET_DIR), random_state=SEED)
manifest.to_csv(ARTIFACT_DIR / "split_manifest.csv", index=False)

class_table = (
    manifest[["class_name", "label"]]
    .drop_duplicates()
    .sort_values("label")
)
CLASS_NAMES = class_table["class_name"].tolist()

print("Images:", len(manifest))
print("Source cases:", manifest["case_id"].nunique())
display(pd.crosstab(manifest["split"], manifest["class_name"]))
display(manifest.groupby("split")["case_id"].unique().to_frame())

In [ ]:
from PIL import Image

examples = manifest.groupby("class_name", group_keys=False).head(1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, (_, row) in zip(axes.flat, examples.iterrows()):
    axis.imshow(Image.open(row["image_path"]).convert("RGB"))
    axis.set_title(row["class_name"])
    axis.axis("off")
plt.suptitle("Kather-2016 tissue classes")
plt.tight_layout()
plt.savefig(PLOT_DIR / "class_examples.png", dpi=180, bbox_inches="tight")
plt.show()

## 2. Scratch-CNN training-size benchmark

Only the training split is subsampled. Validation and test images remain identical across every experiment. Each requested size is distributed as evenly as possible across the eight classes.

In [ ]:
RUN_SCRATCH_BENCHMARK = True
TRAINING_SIZES = (500, 1000, 1500, None)  # None means all grouped-training rows.

benchmark_rows = []
histories = {}
evaluations = {}
models = {}

if RUN_SCRATCH_BENCHMARK:
    for requested_size in TRAINING_SIZES:
        size_name = "full" if requested_size is None else str(requested_size)
        experiment_name = f"scratch_{size_name}"
        experiment_manifest = balanced_training_subset(
            manifest,
            requested_size,
            random_state=SEED,
        )
        actual_size = int((experiment_manifest["split"] == "train").sum())
        print(f"\n{experiment_name}: {actual_size} training images")

        started = time.perf_counter()
        model, history, evaluation = run_benchmark(
            model_builder=lambda: build_report_cnn(num_classes=len(CLASS_NAMES)),
            manifest=experiment_manifest,
            class_names=CLASS_NAMES,
            device=DEVICE,
            checkpoint_path=CHECKPOINT_DIR / f"{experiment_name}.pt",
            random_state=SEED,
            batch_size=SCRATCH_BATCH_SIZE,
            num_workers=NUM_WORKERS,
            epochs=SCRATCH_EPOCHS,
            learning_rate=SCRATCH_LEARNING_RATE,
        )
        runtime_seconds = time.perf_counter() - started
        parameter_counts = count_parameters(model)

        history.to_csv(ARTIFACT_DIR / f"history_{experiment_name}.csv", index=False)
        np.save(
            ARTIFACT_DIR / f"confusion_{experiment_name}.npy",
            evaluation["confusion_matrix"],
        )
        pd.DataFrame(evaluation["classification_report"]).transpose().to_csv(
            ARTIFACT_DIR / f"class_report_{experiment_name}.csv"
        )
        histories[experiment_name] = history
        evaluations[experiment_name] = evaluation
        models[experiment_name] = model
        benchmark_rows.append(
            {
                "experiment": experiment_name,
                "model": "Scratch CNN",
                "requested_training_size": requested_size or actual_size,
                "actual_training_size": actual_size,
                "accuracy": evaluation["accuracy"],
                "balanced_accuracy": evaluation["balanced_accuracy"],
                "macro_f1": evaluation["macro_f1"],
                "test_loss": evaluation["loss"],
                "epochs_completed": len(history),
                "runtime_seconds": runtime_seconds,
                **parameter_counts,
            }
        )
        print(
            f"accuracy={evaluation['accuracy']:.4f}, "
            f"macro-F1={evaluation['macro_f1']:.4f}, "
            f"runtime={runtime_seconds / 60:.1f} min"
        )

## 3. Pretrained ResNet-18 benchmark

This mirrors the report's transfer-learning experiment. It uses all grouped-training images and fine-tunes the complete ImageNet-pretrained network.

In [ ]:
RUN_RESNET18_BENCHMARK = True

if RUN_RESNET18_BENCHMARK:
    experiment_name = "resnet18_full"
    actual_size = int((manifest["split"] == "train").sum())
    started = time.perf_counter()
    model, history, evaluation = run_benchmark(
        model_builder=lambda: build_resnet18(
            num_classes=len(CLASS_NAMES),
            pretrained=True,
            freeze_backbone=False,
        ),
        manifest=manifest,
        class_names=CLASS_NAMES,
        device=DEVICE,
        checkpoint_path=CHECKPOINT_DIR / f"{experiment_name}.pt",
        random_state=SEED,
        batch_size=RESNET_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        epochs=RESNET_EPOCHS,
        learning_rate=RESNET_LEARNING_RATE,
    )
    runtime_seconds = time.perf_counter() - started
    parameter_counts = count_parameters(model)

    history.to_csv(ARTIFACT_DIR / f"history_{experiment_name}.csv", index=False)
    np.save(
        ARTIFACT_DIR / f"confusion_{experiment_name}.npy",
        evaluation["confusion_matrix"],
    )
    pd.DataFrame(evaluation["classification_report"]).transpose().to_csv(
        ARTIFACT_DIR / f"class_report_{experiment_name}.csv"
    )
    histories[experiment_name] = history
    evaluations[experiment_name] = evaluation
    models[experiment_name] = model
    benchmark_rows.append(
        {
            "experiment": experiment_name,
            "model": "ResNet-18",
            "requested_training_size": actual_size,
            "actual_training_size": actual_size,
            "accuracy": evaluation["accuracy"],
            "balanced_accuracy": evaluation["balanced_accuracy"],
            "macro_f1": evaluation["macro_f1"],
            "test_loss": evaluation["loss"],
            "epochs_completed": len(history),
            "runtime_seconds": runtime_seconds,
            **parameter_counts,
        }
    )
    print(
        f"ResNet-18 accuracy={evaluation['accuracy']:.4f}, "
        f"macro-F1={evaluation['macro_f1']:.4f}, "
        f"runtime={runtime_seconds / 60:.1f} min"
    )

## 4. Benchmark summary

In [ ]:
results = pd.DataFrame(benchmark_rows).sort_values(
    ["model", "actual_training_size"]
).reset_index(drop=True)
results.to_csv(ARTIFACT_DIR / "benchmark_summary.csv", index=False)
display(
    results.style.format(
        {
            "accuracy": "{:.4f}",
            "balanced_accuracy": "{:.4f}",
            "macro_f1": "{:.4f}",
            "test_loss": "{:.4f}",
            "runtime_seconds": "{:.1f}",
        }
    )
)

In [ ]:
scratch_results = results[results["model"] == "Scratch CNN"].sort_values(
    "actual_training_size"
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    scratch_results["actual_training_size"],
    scratch_results["accuracy"],
    marker="o",
    label="Accuracy",
)
axes[0].plot(
    scratch_results["actual_training_size"],
    scratch_results["macro_f1"],
    marker="s",
    label="Macro-F1",
)
axes[0].set(
    title="Scratch CNN performance vs. training size",
    xlabel="Grouped-training images",
    ylabel="Score",
    ylim=(0, 1),
)
axes[0].legend()

sns.barplot(data=results, x="experiment", y="macro_f1", hue="model", ax=axes[1])
axes[1].set(title="Benchmark macro-F1", xlabel="Experiment", ylabel="Macro-F1", ylim=(0, 1))
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / "benchmark_performance.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    axes[0].plot(history["epoch"], history["validation_loss"], label=name)
    axes[1].plot(history["epoch"], history["validation_macro_f1"], label=name)
axes[0].set(title="Validation loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].set(title="Validation macro-F1", xlabel="Epoch", ylabel="Macro-F1", ylim=(0, 1))
for axis in axes:
    axis.legend(fontsize=8)
plt.tight_layout()
plt.savefig(PLOT_DIR / "benchmark_learning_curves.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
comparison_names = [
    name for name in ("scratch_full", "resnet18_full") if name in evaluations
]
fig, axes = plt.subplots(1, len(comparison_names), figsize=(7 * len(comparison_names), 5))
if len(comparison_names) == 1:
    axes = [axes]
for axis, name in zip(axes, comparison_names):
    sns.heatmap(
        evaluations[name]["confusion_matrix"],
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axis,
    )
    axis.set(title=name, xlabel="Predicted", ylabel="True")
    axis.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.savefig(PLOT_DIR / "benchmark_confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
run_metadata = {
    "seed": SEED,
    "device": str(DEVICE),
    "class_names": CLASS_NAMES,
    "split_cases": {
        split: sorted(group["case_id"].unique().tolist())
        for split, group in manifest.groupby("split")
    },
    "scratch": {
        "epochs": SCRATCH_EPOCHS,
        "batch_size": SCRATCH_BATCH_SIZE,
        "learning_rate": SCRATCH_LEARNING_RATE,
    },
    "resnet18": {
        "epochs": RESNET_EPOCHS,
        "batch_size": RESNET_BATCH_SIZE,
        "learning_rate": RESNET_LEARNING_RATE,
    },
}
with open(ARTIFACT_DIR / "run_metadata.json", "w", encoding="utf-8") as output_file:
    json.dump(run_metadata, output_file, indent=2)
run_metadata

## Interpretation

The training-size curve measures the value of additional labeled tiles for the same architecture. The full scratch-CNN versus ResNet-18 comparison measures the effect of transfer learning.

These results form the RGB benchmark against which `01_maskcut_cnn_trial.ipynb` should be compared. Because the split is grouped by source case, the resulting accuracy may be lower than the report's original random-split result while providing a more defensible estimate of generalization.